In [2]:
"""
Hardware QGT Proxy — Figures
==============================
Input:  qgt_echo_results.json
Output: fig_echo1_delta_scan.pdf   — P_00(δ), g_θθ(δ): hw vs sim vs analytical
        fig_echo2_gphi_scan.pdf    — g_φφ(θ): hw vs analytical
        fig_echo_panel.pdf         — combined paper figure
"""

import json, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import math

ECHO_JSON = "/Users/nandan/Desktop/CTCs/IBM/qgt_echo_results.json"
OUT_DIR   = "/Users/nandan/Desktop/CTCs/IBM"

with open(ECHO_JSON) as f:
    D = json.load(f)

meta     = D["metadata"]
analyt   = D["analytical"]
sim      = D["simulator"]
hw       = D["hardware"]
shots    = meta["shots"]

deltas      = np.array(hw["g_tt"]["deltas"])
P00_hw      = np.array(hw["g_tt"]["P00_hw"])
P00_lo      = np.array(hw["g_tt"]["P00_lo"])
P00_hi      = np.array(hw["g_tt"]["P00_hi"])
P00_sim     = np.array(hw["g_tt"]["P00_sim"])
g_tt_fit    = hw["g_tt"]["g_fit"]
g_tt_bs_lo  = hw["g_tt"]["g_bs_lo"]
g_tt_bs_hi  = hw["g_tt"]["g_bs_hi"]
g_tt_exact  = analyt["g_tt_exact"]
bs_g        = np.array(hw["g_tt"]["bootstrap_g"])
id_noise    = hw["identity"]["noise_floor"]

thetas_scan = np.array(hw["g_pp"]["thetas"])
g_pp_hw     = np.array(hw["g_pp"]["g_pp_hw"])
g_pp_lo     = np.array(hw["g_pp"]["g_pp_lo"])
g_pp_hi     = np.array(hw["g_pp"]["g_pp_hi"])
g_pp_anal   = np.array(hw["g_pp"]["g_pp_analytical"])

os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 12,
    'axes.titlesize': 12, 'axes.labelsize': 12,
    'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'legend.fontsize': 10, 'figure.dpi': 180,
    'pdf.fonttype': 42,
})
FONT = 12
C_HW = '#C0392B'; C_SIM = '#2980B9'; C_ANAL = '#27AE60'; C_FIT = '#8E44AD'

def apply_style(ax):
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.tick_params(labelsize=FONT, direction='out', length=4,
                   width=0.8, color='#555',
                   bottom=True, left=True, top=False, right=False)

def pi_fmt(val, pos):
    n = val / math.pi
    if abs(n) < 0.01: return '0'
    if abs(n - 0.5) < 0.01: return '$\\pi/2$'
    if abs(n - 1)   < 0.01: return '$\\pi$'
    return f'${n:.2f}\\pi$'

delta_fine = np.linspace(0, max(deltas)*1.08, 300)

def P00_model(d, g):
    return 1.0 - 4.0*g*np.sin(d/2)**2

# =============================================================================
#  FIG 1 — δ-scan
# =============================================================================
fig1, axes1 = plt.subplots(1, 3, figsize=(15, 5.0))

# (a) Raw P_00(δ)
ax = axes1[0]
ax.plot(delta_fine, P00_model(delta_fine, g_tt_exact),
        color=C_ANAL, lw=2.0, ls='--',
        label=f'Analytical ($g={g_tt_exact:.5f}$)', zorder=3)
ax.plot(delta_fine, P00_model(delta_fine, g_tt_fit),
        color=C_FIT,  lw=2.0, ls=':',
        label=f'HW fit ($g={g_tt_fit:.5f}$)', zorder=4)
ax.plot(deltas, P00_sim, color=C_SIM, lw=0, marker='s', ms=7,
        markerfacecolor='none', markeredgewidth=1.5,
        label='Simulator', zorder=5)
ax.errorbar(deltas, P00_hw,
            yerr=[P00_hw-P00_lo, P00_hi-P00_hw],
            fmt='o', color=C_HW, ms=7, lw=1.5, capsize=5, capthick=1.5,
            label=f'Hardware ({shots:,} shots)', zorder=6)
ax.axhline(1-id_noise, color='#888', lw=1.0, ls='-.',
           label=f'Noise floor: $P_{{00}}={1-id_noise:.4f}$')
ax.set_xlabel('$\\delta$', fontsize=FONT)
ax.set_ylabel('$P_{00}(\\delta) = |\\langle\\Psi|\\Psi_\\delta\\rangle|^2$', fontsize=FONT)
ax.set_title('(a) Echo $P_{00}$ vs $\\delta$', fontsize=FONT, pad=8)
ax.legend(fontsize=9, framealpha=0.9)
ax.grid(alpha=0.2); apply_style(ax)

# (b) Derived g_θθ(δ) — should be constant at g_tt_exact
g_hw_per_d  = (1 - P00_hw) / (4*np.sin(deltas/2)**2)
g_lo_per_d  = (1 - P00_hi) / (4*np.sin(deltas/2)**2)
g_hi_per_d  = (1 - P00_lo) / (4*np.sin(deltas/2)**2)
g_sim_per_d = (1 - P00_sim) / (4*np.sin(deltas/2)**2)

ax2 = axes1[1]
ax2.axhline(g_tt_exact, color=C_ANAL, lw=2.5, ls='--',
            label=f'Analytical: {g_tt_exact:.5f}', zorder=3)
ax2.axhspan(g_tt_bs_lo, g_tt_bs_hi, alpha=0.18, color=C_FIT,
            label=f'HW 95% CI [{g_tt_bs_lo:.5f},{g_tt_bs_hi:.5f}]')
ax2.axhline(g_tt_fit, color=C_FIT, lw=2.0, ls=':',
            label=f'HW fit: {g_tt_fit:.5f}', zorder=4)
ax2.scatter(deltas, g_sim_per_d, color=C_SIM, s=50, marker='s',
            zorder=5, facecolors='none', edgecolors=C_SIM, linewidths=1.5,
            label='Simulator point-estimate')
ax2.errorbar(deltas, g_hw_per_d,
             yerr=[g_hw_per_d-g_lo_per_d, g_hi_per_d-g_hw_per_d],
             fmt='o', color=C_HW, ms=7, lw=1.5, capsize=5,
             label='Hardware', zorder=6)
ax2.set_xlabel('$\\delta$', fontsize=FONT)
ax2.set_ylabel('$(1-P_{00}) / [4\\sin^2(\\delta/2)]$', fontsize=FONT)
ax2.set_title('(b) $g_{\\theta\\theta}$ per-$\\delta$ estimate\n'
              '(should be flat = constant)', fontsize=FONT, pad=8)
ax2.legend(fontsize=9, framealpha=0.9)
ax2.grid(alpha=0.2); apply_style(ax2)

# (c) Bootstrap distribution of g
ax3 = axes1[2]
ax3.hist(bs_g, bins=40, color=C_FIT, alpha=0.75, edgecolor='white', lw=0.5)
ax3.axvline(g_tt_exact, color=C_ANAL, lw=2.0, ls='--',
            label=f'Analytical: {g_tt_exact:.5f}')
ax3.axvline(g_tt_fit, color=C_FIT, lw=2.0,
            label=f'HW fit: {g_tt_fit:.5f}')
ax3.axvline(g_tt_bs_lo, color=C_HW, lw=1.2, ls=':', alpha=0.8)
ax3.axvline(g_tt_bs_hi, color=C_HW, lw=1.2, ls=':', alpha=0.8,
            label=f'95% CI')
ax3.set_xlabel('$g_{\\theta\\theta}$', fontsize=FONT)
ax3.set_ylabel('Bootstrap count', fontsize=FONT)
ax3.set_title(f'(c) Bootstrap distribution\n$\\Delta g = {g_tt_fit-g_tt_exact:+.5f}$',
              fontsize=FONT, pad=8)
ax3.legend(fontsize=9, framealpha=0.9)
ax3.grid(alpha=0.2, axis='x'); apply_style(ax3)

fig1.suptitle(
    f'Loschmidt echo $g_{{\\theta\\theta}}$ measurement — ibm_torino\n'
    f'HW fit: $g_{{\\theta\\theta}}={g_tt_fit:.5f}$ [{g_tt_bs_lo:.5f},{g_tt_bs_hi:.5f}],  '
    f'analytical: $g_{{\\theta\\theta}}={g_tt_exact:.5f}$  '
    f'($\\Delta={g_tt_fit-g_tt_exact:+.5f}$)',
    fontsize=FONT, y=1.02
)
fig1.tight_layout()
fig1.savefig(os.path.join(OUT_DIR, 'fig_echo1_delta_scan.pdf'),
             bbox_inches='tight', dpi=200)
plt.close(fig1)
print("[✓] fig_echo1_delta_scan.pdf")

# =============================================================================
#  FIG 2 — g_φφ(θ) scan
# =============================================================================
fig2, ax2f = plt.subplots(figsize=(7.0, 5.0))

ax2f.plot(thetas_scan, g_pp_anal, color=C_ANAL, lw=2.5, ls='-',
          label='Analytical $(1-\\langle\\sigma_z\\rangle^2_{C,\\theta})/4$', zorder=4)
ax2f.errorbar(thetas_scan, g_pp_hw,
              yerr=[g_pp_hw-g_pp_lo, g_pp_hi-g_pp_hw],
              fmt='o', color=C_HW, ms=7, lw=1.5, capsize=5,
              label=f'Hardware ($\\delta\\phi={meta["delta_phi_fixed"]:.2f}$)', zorder=5)

ax2f.set_xlabel('$\\theta$ (base angle)', fontsize=FONT)
ax2f.set_ylabel('$g_{\\phi\\phi}(\\theta)$', fontsize=FONT)
ax2f.set_title(
    r'$g_{\phi\phi}(\theta)$: hardware vs analytical' + '\n'
    r'Rz generator: $(1-\langle\sigma_z\rangle^2_{C,\theta})/4$',
    fontsize=FONT, pad=8
)
ax2f.xaxis.set_major_formatter(FuncFormatter(pi_fmt))
ax2f.legend(fontsize=10, framealpha=0.9)
ax2f.grid(alpha=0.2); apply_style(ax2f)
fig2.tight_layout()
fig2.savefig(os.path.join(OUT_DIR, 'fig_echo2_gphi_scan.pdf'),
             bbox_inches='tight', dpi=200)
plt.close(fig2)
print("[✓] fig_echo2_gphi_scan.pdf")

# =============================================================================
#  FIG 3 — Combined panel
# =============================================================================
fig3 = plt.figure(figsize=(14, 5.0))
gs   = gridspec.GridSpec(1, 3, figure=fig3, wspace=0.40)

ax_p  = fig3.add_subplot(gs[0])
ax_g  = fig3.add_subplot(gs[1])
ax_pp = fig3.add_subplot(gs[2])

# Panel (a) P_00
ax_p.plot(delta_fine, P00_model(delta_fine, g_tt_exact),
          color=C_ANAL, lw=2.0, ls='--', label=f'Analytical')
ax_p.plot(delta_fine, P00_model(delta_fine, g_tt_fit),
          color=C_FIT,  lw=2.0, ls=':', label=f'HW fit')
ax_p.errorbar(deltas, P00_hw,
              yerr=[P00_hw-P00_lo, P00_hi-P00_hw],
              fmt='o', color=C_HW, ms=6, capsize=4, lw=1.5,
              label='Hardware')
ax_p.scatter(deltas, P00_sim, color=C_SIM, s=40, marker='s',
             facecolors='none', edgecolors=C_SIM, linewidths=1.5,
             zorder=5, label='Simulator')
ax_p.set_xlabel('$\\delta$', fontsize=FONT)
ax_p.set_ylabel('$P_{00}(\\delta)$', fontsize=FONT)
ax_p.set_title('(a) Echo $P_{00}$ vs $\\delta$', fontsize=FONT)
ax_p.legend(fontsize=9, framealpha=0.9, loc='lower left')
ax_p.grid(alpha=0.2); apply_style(ax_p)

# Panel (b) g_θθ flat-line check
ax_g.axhline(g_tt_exact, color=C_ANAL, lw=2.5, ls='--',
             label=f'Exact: {g_tt_exact:.5f}')
ax_g.axhspan(g_tt_bs_lo, g_tt_bs_hi, alpha=0.18, color=C_FIT)
ax_g.axhline(g_tt_fit, color=C_FIT, lw=2.0, ls=':',
             label=f'HW fit: {g_tt_fit:.5f}')
ax_g.errorbar(deltas, g_hw_per_d,
              yerr=[g_hw_per_d-g_lo_per_d, g_hi_per_d-g_hw_per_d],
              fmt='o', color=C_HW, ms=6, capsize=4, lw=1.5,
              label='Hardware')
ax_g.set_xlabel('$\\delta$', fontsize=FONT)
ax_g.set_ylabel('$g_{\\theta\\theta}$ estimate', fontsize=FONT)
ax_g.set_title(f'(b) $g_{{\\theta\\theta}}$: flat = uniform\n'
               f'$\\Delta g={g_tt_fit-g_tt_exact:+.5f}$',
               fontsize=FONT)
ax_g.legend(fontsize=9, framealpha=0.9)
ax_g.grid(alpha=0.2); apply_style(ax_g)

# Panel (c) g_φφ(θ)
ax_pp.plot(thetas_scan, g_pp_anal, color=C_ANAL, lw=2.5, ls='-',
           label='Analytical')
ax_pp.errorbar(thetas_scan, g_pp_hw,
               yerr=[g_pp_hw-g_pp_lo, g_pp_hi-g_pp_hw],
               fmt='o', color=C_HW, ms=6, capsize=4, lw=1.5,
               label='Hardware')
ax_pp.set_xlabel('$\\theta$', fontsize=FONT)
ax_pp.set_ylabel('$g_{\\phi\\phi}(\\theta)$', fontsize=FONT)
ax_pp.set_title('(c) $g_{\\phi\\phi}(\\theta)$:\nhw vs analytical', fontsize=FONT)
ax_pp.xaxis.set_major_formatter(FuncFormatter(pi_fmt))
ax_pp.legend(fontsize=9, framealpha=0.9)
ax_pp.grid(alpha=0.2); apply_style(ax_pp)

fig3.suptitle(
    'Hardware QGT proxy: Loschmidt echo on ibm_torino\n'
    f'Exact formula: $P_{{00}}(\\delta)=1-4g_{{\\theta\\theta}}\\sin^2(\\delta/2)$  ·  '
    f'$g_{{\\theta\\theta}}=(1-\\langle\\sigma_x\\rangle^2_C)/4$ (uniform over $\\theta,\\phi$)',
    fontsize=FONT, y=1.02
)
fig3.savefig(os.path.join(OUT_DIR, 'fig_echo_panel.pdf'),
             bbox_inches='tight', dpi=200)
plt.close(fig3)
print("[✓] fig_echo_panel.pdf")

# =============================================================================
#  PAPER-READY NUMBERS
# =============================================================================
print(f"\n{'='*60}")
print("  PAPER-READY NUMBERS")
print(f"{'='*60}")
print(f"  Backend: {meta['backend']},  shots: {shots:,},  DD: {meta.get('dd_sequence','XX')}")
print()
print(f"  Key analytical result:")
print(f"    ⟨σ_x⟩_C = {analyt['sx_C']:.6f}")
print(f"    g_θθ (exact) = {g_tt_exact:.6f}   [independent of (θ,φ)]")
print(f"    Exact formula: P_00(δ) = 1 − 4·{g_tt_exact:.4f}·sin²(δ/2)")
print()
print(f"  Hardware measurement:")
print(f"    g_θθ (NLS fit) = {g_tt_fit:.6f} ± {hw['g_tt']['g_fit_std']:.6f}")
print(f"    g_θθ (bootstrap 95% CI) = [{g_tt_bs_lo:.6f}, {g_tt_bs_hi:.6f}]")
print(f"    Δg = {g_tt_fit - g_tt_exact:+.6f}  "
      f"({100*abs(g_tt_fit-g_tt_exact)/g_tt_exact:.2f}% error)")
print(f"    Noise floor (identity echo): P_00={1-id_noise:.5f}")
print()
print(f"  Analytical conclusion:")
print(f"    The echo circuit U†(θ+δ,φ)·U(θ,φ) simplifies to")
print(f"    part1†·Rx(−δ)·part1, which is independent of (θ,φ).")
print(f"    Therefore Tr Ω is UNIFORM across the parameter space.")
print(f"    This analytically explains the zero Spearman correlation")
print(f"    between Tr Ω and F_msg found in Section V.C.")

[✓] fig_echo1_delta_scan.pdf
[✓] fig_echo2_gphi_scan.pdf
[✓] fig_echo_panel.pdf

  PAPER-READY NUMBERS
  Backend: ibm_torino,  shots: 10,000,  DD: XX

  Key analytical result:
    ⟨σ_x⟩_C = -0.238426
    g_θθ (exact) = 0.235788   [independent of (θ,φ)]
    Exact formula: P_00(δ) = 1 − 4·0.2358·sin²(δ/2)

  Hardware measurement:
    g_θθ (NLS fit) = 0.242299 ± 0.001742
    g_θθ (bootstrap 95% CI) = [0.237161, 0.244042]
    Δg = +0.006511  (2.76% error)
    Noise floor (identity echo): P_00=0.98160

  Analytical conclusion:
    The echo circuit U†(θ+δ,φ)·U(θ,φ) simplifies to
    part1†·Rx(−δ)·part1, which is independent of (θ,φ).
    Therefore Tr Ω is UNIFORM across the parameter space.
    This analytically explains the zero Spearman correlation
    between Tr Ω and F_msg found in Section V.C.
